# RQ2 expected-K=4 probabilistic support preview

CPU-only policy construction. Uses frozen geometry/FLOPs from development seeds 0/1/2, solves geometry and resource-only marginal allocations at exactly Uniform expected interior compute, then constructs maximum-entropy distributions over distinct width pairs. No accuracy is read and no model is trained.

In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
print('Commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip())

## Locate frozen development evidence

In [ ]:
import rq2_anchor_placement
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(RQ2_INPUT, '/kaggle/working/materialized-rq2-support-preview')
print('Frozen development root:', RQ2_ROOT)

## Solve and inspect policies

In [ ]:
from rq2_probabilistic_support import build_support_policy, pair_marginals
OUTPUT_DIR = Path('/kaggle/working/rq2-probabilistic-support-preview')
diagnostics = build_support_policy(RQ2_ROOT, OUTPUT_DIR)
assert diagnostics['training_authorized'] is False
assert all(diagnostics['assertions'].values())
print(json.dumps(diagnostics, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display, Image
marginals = pd.read_csv(OUTPUT_DIR/'support_allocation_marginals.csv')
geometry_q = pd.read_csv(OUTPUT_DIR/'geometry_pair_distribution.csv')
resource_q = pd.read_csv(OUTPUT_DIR/'resource_pair_distribution.csv')
display(marginals)
print('Top geometry pairs:'); display(geometry_q.nlargest(15, 'probability'))
print('Top resource pairs:'); display(resource_q.nlargest(15, 'probability'))
display(Image(filename=str(OUTPUT_DIR/'support_allocation_preview.png')))
print('Starvation review:', json.dumps(diagnostics['starvation_review'], indent=2))
print('POLICY PREVIEW ONLY — inspect pi before authorizing any training.')

In [ ]:
archive = shutil.make_archive('/kaggle/working/rq2-probabilistic-support-preview', 'zip', root_dir=OUTPUT_DIR)
print('Download/persist:', archive)